# Screen Watcher Pro — Jupyter AI Chatbox (spec §12, FR06/T06)

Local web client cho **Tool Watcher API Server**. Client **không chứa logic AI** — chỉ thu thập input,
gửi HTTP request đến server và hiển thị response (spec §12.1).

**Trước khi chạy:** khởi động server trong một terminal:
```
uvicorn app.ai.chat_server:app --host 127.0.0.1 --port 8000 --workers 1
```
(Đặt `ai.mock: true` trong `config/rules.yaml` để demo không cần provider thật,
hoặc `ai.engine: opencode` để đi qua OpenCode CLI — spec §11.)

In [ ]:
# --- 1. Setup: make `app` importable, configure the client ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import requests
from app.ai import chatbox

BASE_URL = "http://127.0.0.1:8000"
USERNAME = "admin"          # demo credentials (seeded admin account)
PASSWORD = "admin123"
SESSION_ID = None            # None -> the server creates a new conversation
print("Client configured for", BASE_URL)

## 2. Health check (FR01, demo scenario §18.1-1)
Server phải trả `status: ok` trước khi demo.

In [ ]:
try:
    r = requests.get(f"{BASE_URL}/health", timeout=10)
    print("HTTP", r.status_code, "->", r.json())
except requests.ConnectionError:
    print("⚠ Server chưa chạy — hãy khởi động uvicorn (xem cell đầu tiên).")

## 3. Đăng nhập lấy JWT token
API yêu cầu Bearer token (bảo mật hơn spec gốc — spec §16 khuyến nghị auth khi mở rộng).

In [ ]:
TOKEN = chatbox.login(BASE_URL, USERNAME, PASSWORD)
print("Đăng nhập OK — token:", TOKEN[:24] + "…")

## 4. Chatbox widget (spec §12.2)
Chat UI bằng `ipywidgets`: history + ô nhập + nút **Send** + toggle *Include latest watcher context*.
Nếu ipywidgets lỗi, tự fallback sang vòng lặp `input()` (spec §19: notebook widget risk mitigation).

In [ ]:
chatbox.launch_chatbox(BASE_URL, token=TOKEN)

## 5. Scripted demo — hỏi trạng thái watcher (demo scenario §18.1-2/4/5)
Gọi thẳng `send_message` (cùng hàm chatbox dùng) để có output lưu lại được trong notebook.

In [ ]:
import uuid
sid = str(uuid.uuid4())   # server requires a UUID session id
reply = chatbox.send_message(BASE_URL, sid, "Trạng thái watcher gần nhất là gì?",
                             token=TOKEN, include_context=True)
print("You: Trạng thái watcher gần nhất là gì?")
print("Bot:", reply)
print("---")

## 6. Latest watcher result (FR07, demo scenario §18.1-3)
`GET /api/watcher/executions/latest` — execution id, OCR text, matched rules, email decisions.

In [ ]:
r = requests.get(f"{BASE_URL}/api/watcher/executions/latest",
                 headers={"Authorization": f"Bearer {TOKEN}"}, timeout=30)
data = r.json()
print("HTTP", r.status_code)
print("has_data      :", data.get("has_data"))
print("execution_id  :", data.get("execution_id"))
print("target_app    :", data.get("target_app"))
print("captured_at   :", data.get("captured_at"))
print("matched_rules :", data.get("matched_rules"))
print("ocr_text      :", (data.get("ocr_text") or "")[:300])

## 7. Trigger watcher thủ công (FR08 — optional)
Chạy capture+OCR+rule ngay (cần cửa sổ Chrome/Edge trên máy chạy server). Bỏ comment để dùng:

In [ ]:
# r = requests.post(f"{BASE_URL}/api/watcher/executions",
#                   json={"targets": ["chrome"], "launch": False},
#                   headers={"Authorization": f"Bearer {TOKEN}"}, timeout=180)
# print(r.status_code, r.json())